<a href="https://colab.research.google.com/github/mandib96/case-analytics-engineer-2026/blob/ajuste_fino/create_trusted_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# importando lib e autenticaçao com cloud
import os
import pandas as pd

from google.oauth2 import service_account
from google.cloud import bigquery

service_account_path = '/content/gcp_service_account.json'
credentials = service_account.Credentials.from_service_account_file(service_account_path)

client_bq = bigquery.Client(credentials=credentials, project=credentials.project_id)

#definindo variaveis
dataset_trusted = 'vendas_trusted'
dataset_analytics = 'vendas_analytics'

tabela_trusted = "tbl_vendas_diaria"

print("Configuração concluida!")

Configuração concluida!


In [15]:
def criar_tabela_trusted():

    # # Cria o Dataset
    # dataset_ref_trusted = client_bq.dataset(dataset_trusted, project=credentials.project_id)
    # new_dataset_trusted = bigquery.Dataset(dataset_ref_trusted)
    # new_dataset_trusted.location = "US" # Define a localização do dataset
    # client_bq.create_dataset(new_dataset_trusted, timeout=30)

    # Cria Tabela Trusted - Camada Intermediaria
    sql_trusted = f"""
    CREATE OR REPLACE TABLE `{credentials.project_id}.{dataset_trusted}.{tabela_trusted}`
    PARTITION BY dt_particao
    AS
      with
      base_raw_granularidade as (

        -- Agrupamento prévio para definir a menor granularidade possível
        select
          id_marca,
          marca,
          id_linha,
          linha,
          data_venda,
          arquivo_origem_gcs,
          data_upload_gcs,
          sum(qtd_venda) as vendas,
          count(*) as linhas_encontradas
        from
          vendas_dataset.tbl_raw_vendas
        group by
          1,2,3,4,5,6,7
      )

      , base_raw_unicidade as (
        select
          id_marca,
          marca,
          id_linha,
          linha,
          data_venda,
          vendas,
          arquivo_origem_gcs,
          data_upload_gcs,

          -- Cria chave unica para identificar linhas com os mesmos valores de marca, linha, data, vendas
          TO_HEX(SHA256(CONCAT(
                          COALESCE(marca, ''),
                          COALESCE(linha, ''),
                          COALESCE(data_venda, ''),
                          CAST(vendas AS STRING)
                          ))) as id_arquivo_raw,

          -- Retorna TRUE para casos onde a granularidade foi normalizada (marcador para registro de que referencia de data possuia mais de 1 ocorrencia na origem)
          Case when linhas_encontradas > 1 then true else false end as flag_granularidade,
        from
          base_raw_granularidade
      )

      , base_ajusta_data as (
        select
          *,
          parse_date('%m/%d/%Y', data_venda) as data_venda_formatada
        from
          base_raw_unicidade
      )

      select
        id_marca,
        marca,
        id_linha,
        linha,
        data_venda_formatada as data_venda,
        EXTRACT(YEAR FROM data_venda_formatada) AS ano,
        EXTRACT(MONTH FROM data_venda_formatada) AS mes,
        EXTRACT(DAY FROM data_venda_formatada) AS dia,
        vendas,
        flag_granularidade,
        row_number() over (partition by id_arquivo_raw order by data_upload_gcs desc) as rn_unicidade,
        data_venda_formatada as dt_particao,
        arquivo_origem_gcs,
        data_upload_gcs,
        current_datetime() as ds
      from
        base_ajusta_data
    """

    try:
        client_bq.query(sql_trusted).result()
        print(f"Tabela {dataset_trusted}.{tabela_trusted} criada com sucesso!")
        return {
            'status': 'success',
            'tabela': tabela_trusted}

    except Exception as e:
        print(f"Erro ao criar tabela: {e}")
        return {
            'status': 'error',
            'error': str(e),
            'tabela': tabela_trusted
        }

# criar_tabela_trusted()

In [3]:
def criar_tabelas_analiticas():

    # # Cria o Dataset
    # dataset_ref_analytics = client_bq.dataset(dataset_analytics, project=credentials.project_id)
    # new_dataset_analytics = bigquery.Dataset(dataset_ref_analytics)
    # new_dataset_analytics.location = "US" # Define a localização do dataset
    # client_bq.create_dataset(new_dataset_analytics, timeout=30)

    # Cria Tabelas Analíticas - Camada Consumo
    queries_analiticas = {
        "tbl_vendas_anomes": f"""
            CREATE OR REPLACE TABLE `{credentials.project_id}.{dataset_analytics}.tbl_vendas_anomes`
            CLUSTER BY ano, mes
            AS

              select
                ano,
                mes,
                SUM(vendas) as vendas
              from
                {dataset_trusted}.{tabela_trusted}
              where
                rn_unicidade = 1
              group by
                1,2
        """,

        "tbl_vendas_marca_linha": f"""
            CREATE OR REPLACE TABLE `{credentials.project_id}.{dataset_analytics}.tbl_vendas_marca_linha`
            CLUSTER BY marca, linha
            AS

              select
                marca,
                linha,
                SUM(vendas) as vendas
              from
                {dataset_trusted}.{tabela_trusted}
              where
                rn_unicidade = 1
              group by
                1,2
        """,

        "tbl_vendas_marca_mensal": f"""
            CREATE OR REPLACE TABLE `{credentials.project_id}.{dataset_analytics}.tbl_vendas_marca_mensal`
            CLUSTER BY ano, mes
            AS

              select
                marca,
                ano,
                mes,
                SUM(vendas) as vendas
              from
                {dataset_trusted}.{tabela_trusted}
              where
                rn_unicidade = 1
              group by
                1,2,3
        """,

        "tbl_vendas_linha_mensal": f"""
            CREATE OR REPLACE TABLE `{credentials.project_id}.{dataset_analytics}.tbl_vendas_linha_mensal`
            CLUSTER BY ano, mes
            AS

              select
                linha,
                ano,
                mes,
                SUM(vendas) as vendas
              from
                {dataset_trusted}.{tabela_trusted}
              where
                rn_unicidade = 1
              group by
                1,2,3
        """
    }

    tabelas_criadas = []
    tabelas_com_erro = []

    # Loop de execuçao
    for tabela, sql in queries_analiticas.items():
        try:
            client_bq.query(sql).result()
            tabelas_criadas.append(tabela)
            print(f"  OK: {tabela}")
        except Exception as e:
            tabelas_com_erro.append(tabela)
            print(f"  ERRO: {tabela} - {str(e)[:100]}")

    status = 'success' if not tabelas_com_erro else ('partial' if tabelas_criadas else 'error')

    return {
        'status': status,
        'tabelas_criadas': tabelas_criadas,
        'tabelas_com_erro': tabelas_com_erro
    }

# criar_tabelas_analiticas()

In [20]:
def executar_camada_trusted_analytics():

    # Execução Camada Trusted

    print("Iniciando ingestão na Camada Trusted[1/2]")

    try:
        resultado_trusted = criar_tabela_trusted()

        if resultado_trusted['status'] == 'error':
            print(f"ERRO: {resultado_trusted.get('error')}")
            print("Pipeline interrompido.")
            return {'status': 'failed', 'trusted': resultado_trusted, 'analytics': None}

        print(f"  OK: {resultado_trusted.get('tabela')}")

    except Exception as e:
        print(f"ERRO: {str(e)}")
        return {'status': 'failed', 'trusted': {'status': 'error', 'error': str(e)}, 'analytics': None}

    # Execução Camada Analytics

    print("Iniciando ingestão na Camada Analytics [2/2]")

    try:
        resultado_analytics = criar_tabelas_analiticas()

        print(f"Tabelas criadas: {len(resultado_analytics['tabelas_criadas'])}/4")

        if resultado_analytics['tabelas_com_erro']:
            print(f"Tabelas com erro: {', '.join(resultado_analytics['tabelas_com_erro'])}")

        status_final = 'success' if resultado_analytics['status'] == 'success' else 'partial'

    except Exception as e:
        print(f"ERRO: {str(e)}")
        resultado_analytics = {'status': 'error', 'error': str(e)}
        status_final = 'partial'

    return {
        'status': status_final,
        'trusted': resultado_trusted,
        'analytics': resultado_analytics
    }

# executar_camada_trusted_analytics()

In [21]:
if __name__ == "__main__":
    resultado = executar_camada_trusted_analytics()


 Iniciando ingestão na Camada Trusted[1/2]
Tabela vendas_trusted.tbl_vendas_diaria criada com sucesso!
  OK: tbl_vendas_diaria

 Iniciando ingestão na Camada Analytics [2/2]
  OK: tbl_vendas_anomes
  OK: tbl_vendas_marca_linha
  OK: tbl_vendas_marca_mensal
  OK: tbl_vendas_linha_mensal
Tabelas criadas: 4/4
